In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x8

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# ============================
# Cell 22: V44 PER-LEAD LENGTH (Ultimate) + Safer YOLO Gate (>=8)
# ============================
import os, gc, csv, glob, math
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# =========================
# 0) CONFIG
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/kaggle/input/physionet-ecg-image-digitization"
TEST_CSV_PATH = f"{DATA_DIR}/test.csv"
SAMPLE_PARQUET_PATH = f"{DATA_DIR}/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

YOLO_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
LEAD_TO_IDX = {n:i for i,n in enumerate(LEAD_NAMES)}

TARGET_H = 256
MAX_W = 2048
YOLO_CONF = 0.12
MAX_CACHE = 12

# ✅ تعديل آمن: بدل 9 خليه 8 حتى ما يصفر مريض كامل إذا YOLO لقط 8 ليدات فقط
YOLO_MIN_DET = 8

print(f"⚡ Device: {DEVICE}")

# =========================
# 1) TEMPLATE IDS
# =========================
if not os.path.exists(SAMPLE_PARQUET_PATH):
    raise FileNotFoundError("❌ sample_submission.parquet not found")

print("📦 Reading template ids...")
template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = template["id"].astype(str).to_numpy()
del template
gc.collect()
print(f"✅ Template rows: {len(template_ids):,}")

# =========================
# 2) test.csv -> per-lead lengths + fs per patient
# =========================
test_len_map = {}   # {pid: {lead: length}}
fs_map = {}         # {pid: fs}

if not os.path.exists(TEST_CSV_PATH):
    raise FileNotFoundError("❌ test.csv not found")

tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
for col in ["id", "lead", "fs", "number_of_rows"]:
    if col not in tdf.columns:
        raise ValueError(f"❌ test.csv missing column: {col}")

for _, r in tdf.iterrows():
    pid = str(r["id"]).strip()
    lead = str(r["lead"]).strip()
    fs = float(r["fs"])
    nrows = int(float(r["number_of_rows"]))
    if pid.endswith(".0"): pid = pid[:-2]
    test_len_map.setdefault(pid, {})[lead] = nrows
    fs_map[pid] = fs

print(f"✅ test.csv parsed: {len(tdf)} rows | unique pids: {len(test_len_map)}")
for pid, d in list(test_len_map.items())[:2]:
    print(f"   pid: {pid} | lens: {d}")

# =========================
# 3) Index images
# =========================
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{DATA_DIR}/**/*.png", recursive=True) + glob.glob(f"{DATA_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

# =========================
# 4) Load models + lead mapping
# =========================
print("⚙️ Loading models...")
yolo_model = None
CLASS_TO_LEAD_IDX = {}

if os.path.exists(YOLO_PATH):
    try:
        yolo_model = YOLO(YOLO_PATH)
        if hasattr(yolo_model, "names"):
            for cid, cname in yolo_model.names.items():
                n = str(cname).strip().replace("Lead", "").replace("lead", "").replace(" ", "")
                n = n.replace("AVR","aVR").replace("AVL","aVL").replace("AVF","aVF")
                if n in LEAD_TO_IDX:
                    CLASS_TO_LEAD_IDX[int(cid)] = LEAD_TO_IDX[n]
        for i in range(12):
            CLASS_TO_LEAD_IDX.setdefault(i, i)
        print(f"✅ YOLO loaded. CLASS_TO_LEAD_IDX: {CLASS_TO_LEAD_IDX}")
    except Exception as e:
        print(f"⚠️ YOLO load failed: {e}")
        yolo_model = None
        CLASS_TO_LEAD_IDX = {i:i for i in range(12)}
else:
    CLASS_TO_LEAD_IDX = {i:i for i in range(12)}

unet_model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    decoder_attention_type="scse"
)
if os.path.exists(UNET_PATH):
    try:
        unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
    except Exception as e:
        print(f"⚠️ UNet state load warning: {e}")
unet_model.to(DEVICE).eval()
print("✅ UNet ready.")

# =========================
# 5) Helpers
# =========================
def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path(pid):
    pid_s = clean_pid(pid)
    p = path_map.get(pid_s)
    if not p:
        try:
            p = path_map.get(str(int(float(pid_s))))
        except:
            p = None
    return p

def preprocess_remove_grid_rgb(img):
    try:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        img = img.copy()
        img[mask > 0] = 255
    except:
        pass
    return img

def robust_grid_spacing_px(img):
    # محاولة تقدير spacing من خطوط الشبكة (قد يفشل)
    try:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        edges = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sx = np.sum(np.abs(edges), axis=0)
        prom = np.percentile(sx, 80)
        peaks, _ = find_peaks(sx, distance=5, prominence=prom)
        if len(peaks) > 5:
            diffs = np.diff(peaks)
            valid = diffs[(diffs > 4) & (diffs < 150)]
            if len(valid) > 4:
                return float(np.median(valid))
    except:
        pass
    return None

def apply_filters(data, fs):
    try:
        data = np.nan_to_num(data)
        nyq = 0.5 * fs
        if nyq <= 0:
            return data
        b, a = butter(3, 0.5/nyq, btype='high')
        return filtfilt(b, a, data)
    except:
        return data

def get_crops_strict(img, model):
    crops = [None]*12
    detected = 0
    if model is not None:
        try:
            h, w = img.shape[:2]
            scale = 1280 / max(h, w)
            if scale < 1:
                img_inf = cv2.resize(img, (int(w*scale), int(h*scale)))
            else:
                img_inf = img
                scale = 1.0

            res = model.predict(img_inf, conf=YOLO_CONF, verbose=False, device=DEVICE)
            if res and res[0].boxes:
                for box in res[0].boxes.data.detach().cpu().numpy():
                    cls = int(box[5])
                    tgt = CLASS_TO_LEAD_IDX.get(cls, cls)
                    if 0 <= tgt < 12:
                        x1, y1, x2, y2 = box[:4] / scale
                        if (x2-x1) > 10 and (y2-y1) > 10:
                            crops[tgt] = img[int(y1):int(y2), int(x1):int(x2)]
                            detected += 1
        except:
            pass
    return crops, detected

def extract_path_fast(prob_map):
    path = np.argmax(prob_map, axis=0)
    # Smooth a bit
    return prob_map.shape[0] - savgol_filter(path, 7, 2)

# Gate جودة خفيف (لا تصفّر كثير)
def lead_quality_ok(sig):
    if sig is None or len(sig) < 50:
        return False
    if not np.all(np.isfinite(sig)):
        return False
    p95 = np.percentile(sig, 95)
    p05 = np.percentile(sig, 5)
    p2p = float(p95 - p05)
    if p2p < 0.03 or p2p > 10.0:
        return False
    d2 = np.diff(sig, 2)
    rough = float(np.mean(np.abs(d2))) / (p2p + 1e-6)
    if rough > 0.55:
        return False
    return True

# =========================
# 6) Compute patient -> outputs in TEST.LEAD LENGTHS
# =========================
patient_cache = OrderedDict()

def compute_patient(pid):
    pid = clean_pid(pid)

    # We must know lengths for this patient from test.csv
    lead_lens = test_len_map.get(pid)
    if lead_lens is None:
        # no metadata -> safest zeros using max default 5000
        return {lead: np.zeros(5000, dtype=np.float32) for lead in LEAD_NAMES}

    # init dict of outputs per lead with exact lengths
    out = {lead: np.zeros(int(lead_lens.get(lead, 5000)), dtype=np.float32) for lead in LEAD_NAMES}

    path = get_image_path(pid)
    if not path:
        return out

    try:
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            return out
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        fs = float(fs_map.get(pid, 500.0))

        # 1) YOLO crops
        crops, det_count = get_crops_strict(img, yolo_model)

        # ✅ Safer gate: require >=8 (was 9)
        if det_count < YOLO_MIN_DET:
            return out

        # fallback grid crops for missing
        h, w = img.shape[:2]
        rh, cw = h//4, w//3
        grid_crops = [img[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]
        for i in range(12):
            if crops[i] is None:
                crops[i] = grid_crops[i]

        global_grid = robust_grid_spacing_px(img)

        # 2) Batch prep for UNet
        inputs = []
        scales = []
        clean_crops = []
        for i in range(12):
            c = preprocess_remove_grid_rgb(crops[i])
            clean_crops.append(c)

            ch, cw0 = c.shape[:2]
            s = TARGET_H / ch
            nw = int(cw0 * s)
            if nw > MAX_W:
                nw = MAX_W
            rz = cv2.resize(c, (nw, TARGET_H))
            t = torch.from_numpy(rz).permute(2,0,1).float()/255.0
            inputs.append(t)
            scales.append(s)

        mw = max([t.shape[2] for t in inputs])
        mw = int(np.ceil(mw/32)*32)
        batch = torch.zeros((12, 3, TARGET_H, mw), device=DEVICE)
        for i, t in enumerate(inputs):
            batch[i, :, :, :t.shape[2]] = t.to(DEVICE)

        with torch.no_grad():
            preds = torch.sigmoid(unet_model(batch)).cpu().numpy()

        # 3) Extract each lead then resample to its own target length
        for li, lead in enumerate(LEAD_NAMES):
            w_orig = inputs[li].shape[2]
            prob = preds[li, 0, :, :w_orig]
            raw = extract_path_fast(prob)

            local_grid = robust_grid_spacing_px(clean_crops[li])
            grid_px = local_grid if local_grid else global_grid

            # fallback constant if grid fails
            if grid_px is None or grid_px < 5:
                ppmv = 25.0 * 2.0 * scales[li]
            else:
                ppmv = grid_px * 2.0 * scales[li]  # assume big-box (5mm=0.5mV)

            if ppmv < 5:
                ppmv = 50.0

            sig_mv = (raw - np.median(raw)) / ppmv
            sig_mv = sig_mv - np.median(sig_mv)
            sig_mv = apply_filters(sig_mv, fs)

            if lead_quality_ok(sig_mv):
                tlen = int(lead_lens.get(lead, len(out[lead])))
                out[lead] = resample(sig_mv, tlen).astype(np.float32)
            # else: keep zeros

        # Einthoven consistency mild blend if available
        if len(out['I']) > 0 and len(out['II']) > 0 and len(out['III']) > 0:
            # resample I and III to II length for blending
            L2 = len(out['II'])
            I2 = resample(out['I'], L2) if len(out['I']) != L2 else out['I']
            III2 = resample(out['III'], L2) if len(out['III']) != L2 else out['III']
            out['II'] = (0.6*out['II'] + 0.4*(I2 + III2)).astype(np.float32)

        return out

    except:
        torch.cuda.empty_cache()
        return out

# =========================
# 7) Write submission.csv using PER-LEAD LENGTHS
# =========================
print("🚀 Writing submission.csv (V44 PER-LEAD LENGTH + YOLO>=8)...")

with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing"):
        try:
            parts = rid.rsplit("_", 2)
            pid = clean_pid(parts[0])
            idx = int(parts[1])
            lead = parts[2]

            # cache per patient
            if pid in patient_cache:
                out_dict = patient_cache[pid]
                patient_cache.move_to_end(pid)
            else:
                out_dict = compute_patient(pid)
                patient_cache[pid] = out_dict
                if len(patient_cache) > MAX_CACHE:
                    patient_cache.popitem(last=False)

            arr = out_dict.get(lead)
            val = 0.0
            if arr is not None and 0 <= idx < len(arr):
                v = float(arr[idx])
                if np.isfinite(v):
                    val = v

            writer.writerow([rid, f"{val:.4f}"])

        except:
            writer.writerow([rid, "0.0000"])

gc.collect()
torch.cuda.empty_cache()
print("✅ Done. submission.csv ready.")


⚡ Device: cuda
📦 Reading template ids...
✅ Template rows: 75,000
✅ test.csv parsed: 24 rows | unique pids: 2
   pid: 1053922973 | lens: {'I': 2500, 'II': 10000, 'III': 2500, 'aVR': 2500, 'aVL': 2500, 'aVF': 2500, 'V1': 2500, 'V2': 2500, 'V3': 2500, 'V4': 2500, 'V5': 2500, 'V6': 2500}
   pid: 2352854581 | lens: {'I': 2500, 'II': 10000, 'III': 2500, 'aVR': 2500, 'aVL': 2500, 'aVF': 2500, 'V1': 2500, 'V2': 2500, 'V3': 2500, 'V4': 2500, 'V5': 2500, 'V6': 2500}
🗂️ Indexing images...
✅ Indexed images: 8,795
⚙️ Loading models...
✅ YOLO loaded. CLASS_TO_LEAD_IDX: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}
✅ UNet ready.
🚀 Writing submission.csv (V44 PER-LEAD LENGTH + YOLO>=8)...


Processing: 100%|██████████| 75000/75000 [00:03<00:00, 21328.01it/s]


✅ Done. submission.csv ready.


In [5]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.94 MB
🎉 Ready to Submit.
